# PPOCRv4 fine-tuned rec → ONNX

把主線專案 `output_rec/best_accuracy.pdparams`（acc 91.86%）轉成 RapidOCR 可吃的 ONNX。

輸出：`/content/rec_finetuned.onnx` → 下載到 `yoloOcr/models/rec_finetuned.onnx`（覆蓋）

In [ ]:
# 1. 安裝環境（CPU 版 paddle，轉換不需要 GPU）
!pip install paddlepaddle==3.0.0 -q
!python -c "import paddle; print('paddle OK:', paddle.__version__)"
!pip install "paddle2onnx>=1.3.0" onnxruntime -q
!python -c "import paddle2onnx; print('paddle2onnx OK')"

In [ ]:
# 2. Clone 兩個 repo：trainingOcr（含 config）+ PaddleOCR（含 export_model.py 與字典）
!git clone https://github.com/install88/trainingOcr.git /content/trainingOcr
!git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git /content/PaddleOCR
%cd /content/trainingOcr
!ls configs/rec/ && ls /content/PaddleOCR/tools/export_model.py /content/PaddleOCR/ppocr/utils/ppocr_keys_v1.txt

In [ ]:
# 3. 從 Google Drive 拿 model.pdparams（推薦）
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

# 主線專案 best_accuracy.pdparams 路徑（在你的 Drive ocr_project/output_rec/ 內）
SRC = '/content/drive/MyDrive/ocr_project/output_rec/best_accuracy.pdparams'
DST = '/content/trainingOcr/model.pdparams'

if not os.path.exists(SRC):
    # 若路徑不同，列出 output_rec/ 內容方便你找
    print('預設路徑不存在，列出 output_rec/ 內容：')
    !ls /content/drive/MyDrive/ocr_project/output_rec/
    raise FileNotFoundError(f'找不到 {SRC}，請修改 SRC 變數指向正確位置')

shutil.copy(SRC, DST)
print('複製完成:', os.path.getsize(DST) / 1e6, 'MB')

In [ ]:
# 3.alt（若 Drive 找不到，可用手動上傳）
# from google.colab import files
# uploaded = files.upload()   # 選 best_accuracy.pdparams
# import shutil, os
# fname = next(iter(uploaded))
# shutil.move(fname, '/content/trainingOcr/model.pdparams')
# print('上傳完成:', os.path.getsize('/content/trainingOcr/model.pdparams') / 1e6, 'MB')

In [ ]:
# 4. 用官方 PaddleOCR export 工具
!pip install lmdb -q

import os, re

# 把 config 裡的 hardcode Windows 字典路徑換成 Colab 路徑
config_path = '/content/trainingOcr/configs/rec/PP-OCRv4_mobile_rec_finetune.yml'
with open(config_path, 'r') as f:
    cfg = f.read()

cfg = re.sub(
    r'character_dict_path:.*',
    'character_dict_path: /content/PaddleOCR/ppocr/utils/ppocr_keys_v1.txt',
    cfg
)

with open(config_path, 'w') as f:
    f.write(cfg)

print('config 路徑已修正：')
!grep character_dict_path {config_path}

# Export
!python /content/PaddleOCR/tools/export_model.py \
    -c {config_path} \
    -o Global.pretrained_model=/content/trainingOcr/model.pdparams \
       Global.save_inference_dir=/content/inference_rec

print('\n--- inference_rec contents ---')
!ls -lh /content/inference_rec/

In [ ]:
# 5. 轉 ONNX（PaddlePaddle 3.x 用 inference.json 而非 inference.pdmodel）
!paddle2onnx \
    --model_dir /content/inference_rec \
    --model_filename inference.json \
    --params_filename inference.pdiparams \
    --save_file /content/rec_finetuned.onnx \
    --opset_version 11

import os
size_mb = os.path.getsize('/content/rec_finetuned.onnx') / 1024 / 1024
print(f'ONNX size: {size_mb:.1f} MB')

In [ ]:
# 6. 驗證 input shape
import onnxruntime as ort
import numpy as np

sess = ort.InferenceSession('/content/rec_finetuned.onnx')
inp = sess.get_inputs()[0]
print(f'Input : {inp.name}  shape={inp.shape}  dtype={inp.type}')
print(f'Outputs: {[o.name for o in sess.get_outputs()]}')

dummy = np.zeros([1, 3, 48, 320], dtype=np.float32)
out = sess.run(None, {inp.name: dummy})
print(f'Output shape: {out[0].shape}')
print('ONNX OK')

In [ ]:
# 7. 備份到 Drive + 下載到本機
import shutil
shutil.copy('/content/rec_finetuned.onnx',
            '/content/drive/MyDrive/ocr_project/rec_finetuned_91p86.onnx')
print('已備份到 Drive: MyDrive/ocr_project/rec_finetuned_91p86.onnx')

from google.colab import files
files.download('/content/rec_finetuned.onnx')
# 下載完放到：yoloOcr/models/rec_finetuned.onnx（覆蓋）